# C1 — Une image est un vecteur

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| C — Reconnaître des chiffres manuscrits | 1 sur 4 | 2 h 30 | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Dans les années 1990, les services postaux voulaient lire automatiquement les codes postaux écrits à la main sur les enveloppes. Aujourd'hui, on confie ce genre de tâche à des réseaux de neurones, entraînés sur des millions d'exemples, dont le fonctionnement est difficile à expliquer.

Ce projet pose une question plus modeste et plus instructive :

**Peut-on reconnaître un chiffre écrit à la main sans réseau de neurones, avec les seuls outils de l'algèbre linéaire de ce parcours ? Jusqu'où, et à quel prix ?**

Les données sont 1 797 chiffres manuscrits numérisés en images de 8 × 8 pixels, chaque pixel valant de 0 (blanc) à 16 (noir), avec le chiffre réellement écrit.

![Schéma : les quatre étapes du projet C, de C1 une image est un vecteur à C4 évaluer et livrer, avec les modules réinvestis](../../../assets/schemas/schema-projet-c-etapes.svg)

**Activité** : Appliquez la première étape du processus du parcours : formulez la question de façon précise. Que veut dire « reconnaître » ? Quel taux de réussite faudrait-il pour qu'une machine soit utile au tri du courrier ? Qu'est-ce qu'une erreur coûte ?

Indiquez vos réponses et justifiez-les.

**Activité** : Découpez le problème en sous-problèmes : représenter une image, comparer deux images, décider, évaluer. Pour chacun, notez une idée de méthode.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from chiffres import charger, decouper, afficher, matrice_confusion

X, y = charger()
print(f"{X.shape[0]} images de {X.shape[1]} pixels ; effectifs par chiffre : {np.bincount(y)}")
afficher(X[:30], titres=y[:30])
plt.show()

In [ ]:
pari_reussite_chiffre_moyen = 0.6    # part des chiffres reconnus par la méthode la plus simple (modifiez)

## ③ Abstraire : une image est un vecteur

On range les 64 pixels d'une image, ligne après ligne, dans un vecteur $\mathbf{x}$ de $\mathbb{R}^{64}$. Toute l'algèbre linéaire du parcours devient alors disponible : longueurs, distances, angles, moyennes, projections.

![Schéma : une image de 8 sur 8 pixels est aplatie en un vecteur de 64 nombres ; deux images se ressemblent si leurs vecteurs sont proches](../../../assets/schemas/schema-image-vecteur.svg)

**Une première méthode : le chiffre moyen.** Pour chaque chiffre $c$, on calcule la moyenne $\mathbf{m}_c$ de toutes ses images d'apprentissage : un « 3 typique ». Pour lire une nouvelle image $\mathbf{x}$, on répond le chiffre dont la moyenne est la plus proche :

$$
\hat c = \arg\min_c \lVert \mathbf{x} - \mathbf{m}_c \rVert .
$$

**Évaluer honnêtement.** Comme dans les modules M6 et les projets précédents, on met de côté un paquet de **test** que la méthode ne voit pas pendant sa construction.

**Activité** : À quoi sert cette abstraction ? Qu'est-ce que le vecteur garde de l'image, et qu'est-ce qu'il perd ? Par exemple, sait-il que deux pixels sont voisins sur l'image ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Découper les données

In [ ]:
X_app, y_app, X_test, y_test = decouper(X, y, part_test=0.25, graine=0)
print(f"apprentissage : {len(y_app)} images ; test : {len(y_test)} images")

### Les chiffres moyens

In [ ]:
moyennes = np.array([X_app[y_app == c].mean(axis=0) for c in range(10)])
afficher(moyennes, titres=[f"moyenne : {c}" for c in range(10)], taille=1.5)
plt.show()

distances_moyennes = np.linalg.norm(moyennes[:, None, :] - moyennes[None, :, :], axis=2)
np.fill_diagonal(distances_moyennes, np.inf)
i, j = np.unravel_index(np.argmin(distances_moyennes), distances_moyennes.shape)
print(f"Les deux chiffres moyens les plus proches : {i} et {j} (distance {distances_moyennes[i, j]:.1f})")

### Reconnaître par le chiffre moyen le plus proche

In [ ]:
def predire_moyenne(X_nouveaux, moyennes):
    distances = np.linalg.norm(X_nouveaux[:, None, :] - moyennes[None, :, :], axis=2)   # une ligne par image
    return np.argmin(distances, axis=1)

predits = predire_moyenne(X_test, moyennes)
reussite = np.mean(predits == y_test)
print(f"Taux de réussite sur le test : {reussite:.1%} ({np.sum(predits != y_test)} erreurs sur {len(y_test)})")

M = matrice_confusion(y_test, predits)
fig, ax = plt.subplots(figsize=(5.5, 4.8))
ax.imshow(M, cmap="liouai")
for a in range(10):
    for b in range(10):
        if M[a, b]:
            ax.text(b, a, M[a, b], ha="center", va="center", fontsize=8, color="white" if M[a, b] > M.max() / 2 else LIOUAI["navy"])
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("chiffre reconnu"); ax.set_ylabel("chiffre écrit"); ax.set_title("Matrice de confusion")
plt.tight_layout(); plt.show()

### Regarder les erreurs

In [ ]:
erreurs = np.flatnonzero(predits != y_test)[:20]
afficher(X_test[erreurs], titres=[f"{y_test[k]} lu {predits[k]}" for k in erreurs])
plt.show()

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_reussite_chiffre_moyen:.0%} ; résultat : {reussite:.1%}")

**Activité** :

1. Le chiffre moyen fait-il mieux ou moins bien que vous ne pensiez ? Ce taux serait-il suffisant pour trier du courrier ?
2. Quelles confusions sont les plus fréquentes ? Sont-elles compréhensibles pour un lecteur humain ?
3. Parmi les erreurs affichées, lesquelles auriez-vous faites vous-même ?
4. Rédigez la suite du travail : quelles idées pour faire mieux, dans quel ordre les essayer ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Un modèle trop pauvre.** Un seul « 7 moyen » ne peut pas représenter à la fois les 7 barrés et les 7 simples. Comment tenir compte de la variété des écritures ? (C'est l'objet des étapes C2 et C3.)
- **L'ordre des pixels.** Mélangez les 64 colonnes de toutes les images **avec la même permutation** : les images deviennent illisibles pour un humain. Que devient le taux de réussite de la méthode ? Qu'en concluez-vous sur ce que la méthode « voit » ?
- **Généraliser.** Écrivez une fonction `reussite_par_chiffre(vrais, predits)` qui renvoie le taux de réussite de chaque chiffre, et utilisez-la pour faire l'expérience du mélange des pixels.

In [ ]:
def reussite_par_chiffre(vrais, predits):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def reussite_par_chiffre(vrais, predits):
    return np.array([np.mean(predits[vrais == c] == c) for c in range(10)])

print("Réussite par chiffre :", reussite_par_chiffre(y_test, predits).round(2))

melange = np.random.default_rng(1).permutation(64)
moyennes_m = np.array([X_app[y_app == c][:, melange].mean(axis=0) for c in range(10)])
predits_m = predire_moyenne(X_test[:, melange], moyennes_m)
print(f"Pixels mélangés : réussite {np.mean(predits_m == y_test):.1%}, identique ? {np.array_equal(predits_m, predits)}")
afficher(X_test[:5, melange], titres=y_test[:5])
plt.show()
print("La distance ne dépend pas de l'ordre des coordonnées : la méthode ignore la forme spatiale de l'image.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Découper un gros problème en petits problèmes": 1,
    "Formuler des questions précises": 1,
    "Appliquer le processus en 4 étapes": 1,
    "Estimer la solution avant de calculer": 1,
    "Identifier le but de l'abstraction": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "C1-cadrer-et-explorer", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")